# PPO — One Real PPO Step, Walked Through

This notebook narrates `PPOTrainer.ppo_step()`'s real 10-step call sequence (see
`ppo_trainer/ppo_trainer.py` and Figure 5 in the README) one cell at a time, calling the
actual classes this repo ships — nothing here is reimplemented, everything is imported
straight from `config`, `base_setup`, `models`, `ppo_trainer`, `reward_kl`, and `advantage`.

**Honest framing up front** (expanded again at the end): this is **one PPO step**, not a
training loop — there is no multi-epoch loop anywhere in this repo. The reward model and the
"SFT model" are both architecturally present but never actually trained (see README Section 6)
— treat every number below as "the plumbing runs correctly on real tensors," not "the policy is
learning anything meaningful yet."


In [1]:
import torch

from config import PPOConfig
from ppo_trainer import PPOTrainer

torch.manual_seed(0)

## Step 0 — Configuration and trainer construction

`PPOConfig` holds the model name, LoRA settings, and PPO hyperparameters. Building `PPOTrainer`
loads three copies of `HuggingFaceTB/SmolLM2-135M-Instruct` (policy, SFT/reference, value —
each wrapped with its own LoRA adapter or head) plus the OpenAssistant reward model via `BaseModelSetup`, which is also where the two Pass 2a
fixes live: `model_setup.py` forces `dtype=torch.float32` and `tokenizer_setup.py` sets
`padding_side="left"`.


In [2]:
config = PPOConfig()
trainer = PPOTrainer(config)

print(f"device: {trainer.device}")
print(f"model_name: {config.model_name}")
print(f"tokenizer padding_side: {trainer.base_setup.tokenizer.padding_side}")
print(f"policy_model dtype: {next(trainer.policy_model.parameters()).dtype}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

device: mps
model_name: HuggingFaceTB/SmolLM2-135M-Instruct
tokenizer padding_side: left
policy_model dtype: torch.float32


## Step 1 — Generate sequences and extract (state, action) pairs

`SequenceGenerator.generate_sequences()` samples completions from the policy model for a batch
of prompts and, per generated token, records the (state, action) pair plus the policy's and the
reference (SFT) model's log-probabilities for that token — the raw material every later step
builds on.


In [3]:
prompts = [
    "SUBREDDIT: r/relationships\nTITLE: Should I admit to snooping?\nPOST: ...\nTL;DR:",
    "SUBREDDIT: r/advice\nTITLE: Need help with decision\nPOST: ...\nTL;DR:",
]

sa_pairs, generated_texts = trainer.generator.generate_sequences(prompts)

print(f"sa_pairs: {len(sa_pairs)} (state, action) pairs across {len(prompts)} prompts")
print(f"generated_texts: {generated_texts}")

sa_pairs: 20 (state, action) pairs across 2 prompts
generated_texts: [' How would you feel if you could tell the other', ' If you don’t know who you’re making love']


## Step 2 — Add KL divergence per token

`KLDivergence.add_kl_to_sa_pairs()` computes `policy_logprob - ref_logprob` for every (s, a)
pair — this is the per-token KL penalty term that keeps the policy from drifting too far from
the frozen reference (SFT) model.


In [4]:
sa_pairs = trainer.kl_divergence.add_kl_to_sa_pairs(sa_pairs)

first = sa_pairs[0]
print(f"first pair action_token: {first['action_token']!r}")
print(f"first pair kl_div: {first['kl_div']:.4f}")

first pair action_token: ' How'
first pair kl_div: 0.4714


## Step 3 — Compute rewards

`RewardComputer.compute_rewards()` scores the full (prompt + generated completion) text with
the reward model — one scalar reward per prompt in the batch.


In [5]:
rewards = trainer.reward_computer.compute_rewards(prompts, generated_texts)

print(f"rewards: {rewards}")

rewards: [0.8822776675224304, 1.9822046756744385]


## Step 4 — Adjust rewards with the KL penalty

`RewardAdjuster.adjust_rewards_with_kl()` subtracts `kl_coef * kl_div` from each pair's
sequence-level reward, giving every (s, a) pair its own KL-penalized `adjusted_reward`.


In [6]:
sa_pairs = trainer.reward_adjuster.adjust_rewards_with_kl(sa_pairs, rewards, config.kl_coef)

print(f"kl_coef: {config.kl_coef}")
print(f"first pair adjusted_reward: {sa_pairs[0]['adjusted_reward']:.4f}")

kl_coef: 0.1
first pair adjusted_reward: 0.8351


## Step 5 — Add value predictions

`ValueAdder.add_values_to_sa_pairs()` runs the value model over every state and attaches its
scalar value prediction to the corresponding (s, a) pair.


In [7]:
sa_pairs = trainer.value_adder.add_values_to_sa_pairs(sa_pairs)

print(f"first pair value: {sa_pairs[0]['value']:.4f}")

first pair value: -0.7833


## Step 6 — Compute TD errors

`TDError.compute_td_errors()` groups pairs by prompt, walks each prompt's generated sequence in
order, and computes `td_error = adjusted_reward + gamma * next_value - value` per step (using
0 for the bootstrap value after the last generated token).


In [8]:
sa_pairs = trainer.td_error.compute_td_errors(sa_pairs, config.gamma)

print(f"gamma: {config.gamma}")
print(f"first pair td_error: {sa_pairs[0]['td_error']:.4f}")

gamma: 0.99
first pair td_error: 4.2198


## Step 7 — Compute GAE advantages

`GAEAdvantage.compute_gae_advantages()` walks each prompt's sequence backwards, accumulating
the generalized advantage estimate `gae = td_error + gamma * lam * gae` at each step.


In [9]:
sa_pairs = trainer.gae_advantage.compute_gae_advantages(sa_pairs, config.gamma, config.lam)

print(f"lam: {config.lam}")
print(f"first pair advantage: {sa_pairs[0]['advantage']:.4f}")

lam: 0.95
first pair advantage: 7.4572


## Step 8 — Compute reward-to-go (return)

`ReturnCalculator.add_reward_to_go()` sets each pair's return target for the value model as
`advantage + value` — the standard GAE return reconstruction.


In [10]:
sa_pairs = trainer.return_calculator.add_reward_to_go(sa_pairs)

print(f"first pair reward_to_go: {sa_pairs[0]['reward_to_go']:.4f}")

first pair reward_to_go: 6.6739


## Step 9 — Train the value model

`ModelTrainer.train_value_model()` takes one gradient step on the value model, regressing its
predictions toward each pair's `reward_to_go` target with MSE loss.


In [11]:
value_loss = trainer.model_trainer.train_value_model(sa_pairs)

print(f"value_loss: {value_loss:.4f}")

value_loss: 64.6847


## Step 10 — Train the policy model

`ModelTrainer.train_policy_model()` recomputes the policy's current log-probabilities for each
recorded action, forms the PPO-clip ratio against the old log-probabilities captured back in
Step 1, and takes one gradient step on the clipped surrogate policy loss.


In [12]:
policy_loss = trainer.model_trainer.train_policy_model(sa_pairs)

print(f"policy_loss: {policy_loss:.4f}")

policy_loss: -5.4826


## Recap — the full `ppo_step()` output

This is the dict `PPOTrainer.ppo_step(prompts)` returns in one call (as `make train` does)
— we've just walked through the ten steps that build it individually above.


In [13]:
results = {
    "sa_pairs": sa_pairs,
    "generated_texts": generated_texts,
    "rewards": rewards,
    "value_loss": value_loss,
    "policy_loss": policy_loss,
}

print("Generated texts:", results["generated_texts"])
print("Rewards:", results["rewards"])
print(f"Value loss: {results['value_loss']:.4f}")
print(f"Policy loss: {results['policy_loss']:.4f}")

Generated texts: [' How would you feel if you could tell the other', ' If you don’t know who you’re making love']
Rewards: [0.8822776675224304, 1.9822046756744385]
Value loss: 64.6847
Policy loss: -5.4826


## Caveats — what this notebook does and doesn't demonstrate

- **This is one PPO step, not a training run.** There is no multi-epoch loop anywhere in this
  repo (`training/run_ppo.py` and this notebook both call `ppo_step()` once); running this cell
  again would take one more gradient step from wherever the models currently sit, not resume a
  tracked training curve.
- **The reward model is pretrained, not trained here.** Reward scoring uses the OpenAssistant
  DeBERTa-v3 model (`OpenAssistant/reward-model-deberta-v3-large-v2`) via `RewardComputer`; the
  in-repo `RewardModel` (a fresh scalar head) is kept only for the component tests. The `rewards`
  in Step 3 are a real quality signal, but they score an untrained policy.
- **The "SFT model" was never actually fine-tuned.** `SFTModel` is architecturally a fresh,
  independently LoRA-initialized copy of the base model — a valid *fixed* KL-reference point
  (it never gets optimized during the PPO step), but not the output of a real supervised
  fine-tuning stage.
- **What *is* real:** the generation, KL computation, reward adjustment, GAE advantage, and
  both loss computations above all ran on real tensors through real forward/backward passes —
  the PPO plumbing itself is correct, even though the signal driving it (the reward model) isn't
  meaningful yet. Building a real reward-model training stage is flagged as a Pass 3 candidate
  in `TODO.md`, not done here.
